In [1]:
# Pipeline riemanniana a banco di filtri con riallineamento per run (run 4, 8, 12).
# Sinistra vs destra immaginata, stessa impalcatura delle altre pipeline per essere
# direttamente confrontabile: stesse run, finestre 2s/0.5s, stessa Leave-One-Run-Out,
# stessa soglia probabilistica, stessi canali. Cambia solo come si estraggono le feature
# e quale classificatore le usa.
#
# Perche' questo approccio e non una rete neurale: per fold ci sono circa 28 trial
# indipendenti. In quel regime una rete addestrata da zero per soggetto non generalizza,
# mentre i metodi basati su covarianza con regolarizzazione a shrinkage sono progettati
# esattamente per il caso "poche osservazioni, molte dimensioni".
# --- Preambolo standard ------------------------------------------------------
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "dataset_description.json").exists()
)
DATA  = PROJECT_ROOT / "data"
PLOTS = PROJECT_ROOT / "src" / "plots"
sys.path.insert(0, str(PROJECT_ROOT / "src"))
# -----------------------------------------------------------------------------

import time
import warnings
import numpy as np
import mne
from mne_bids import BIDSPath, read_raw_bids
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.exceptions import ConvergenceWarning
from util.preprocessing import create_sliding_windows, create_window_labels
from util.riemann_filterbank import (
    DEFAULT_BANDS, run_covariances, reference_mean, recenter, to_tangent, n_features
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)
mne.set_log_level('WARNING')

# --- Configurazione ----------------------------------------------------------

root = DATA
runs = ["6", "10", "14"]
test_run_order = [runs[2], runs[1], runs[0]]
first_person = 1
people = 109

window_size = 2         # identici alle altre pipeline, per confrontabilita'
step_size = 0.5
label_threshold = 0.8

# Banda larga: le sotto-bande del banco fanno la selezione fine
L_FREQ = 4
H_FREQ = 40
BANDS = DEFAULT_BANDS   # [(8, 13), (13, 20), (20, 30)]

CHANNELS = ["C3", "C4", "Cz", "Fc3", "Fc4", "Fcz", "Cp3", "Cp4", "Cpz"]

# Riallineamento delle covarianze:
#   "per_run" -> ogni run viene riportata sul proprio baricentro. Il baricentro non usa le
#                etichette, quindi si puo' stimare anche sulla run di test: e' adattamento di
#                dominio non supervisionato. E' la configurazione che ci si aspetta migliore.
#   "train"   -> tutte le run vengono riportate sul baricentro delle sole run di training.
#                Piu' conservativo: nessuna statistica della run di test viene usata.
RECENTER = "per_run"

# Il classificatore e' quasi privo di iperparametri: la LDA con shrinkage di Ledoit-Wolf
# regolarizza da sola ed e' deterministica, a differenza dell'SVM con probability=True le cui
# probabilita' passano da una calibrazione di Platt con mescolamento casuale interno.
# Griglia minuscola: con 28 trial indipendenti cercare fra decine di combinazioni insegue rumore.
param_grid = {
    "clf__shrinkage": ["auto", 0.2, 0.5],
}

START_THRESHOLD = 0.90
MIN_THRESHOLD = 0.50
MIN_ACCEPTED_RATIO = 0.70

# NaN e non 0: un fold che fallisce deve restare fuori dalle medie, non entrarci come 0%
all_accuracy = np.full((people, len(test_run_order)), np.nan)
all_discarded = np.full((people, len(test_run_order)), np.nan)
cm_sum = np.zeros((2, 2))

print(f"Feature per finestra: {n_features(len(CHANNELS), len(BANDS))} "
      f"({len(BANDS)} bande x {len(CHANNELS)} canali)")
print()

# --- Loop principale ---------------------------------------------------------

for i in range(first_person, first_person + people):
    subject = f"{i:03d}"
    subject_start = time.time()

    print("=" * 60)
    print(f"Paziente {subject}")
    print("=" * 60)

    for test_index, test_run in enumerate(test_run_order):
        test_start = time.time()
        train_runs = [run for run in runs if run != test_run]

        # Per ogni run si conservano le covarianze di TUTTE le finestre (servono a stimare il
        # baricentro senza usare le etichette) e la maschera di quelle attive.
        per_run = {}
        trial_offset = 0

        for run in runs:
            bids_path = BIDSPath(
                subject=subject, task="motion", run=run, datatype="eeg", root=root,
            )

            try:
                raw = read_raw_bids(bids_path, verbose=False)
                events, event_id = mne.events_from_annotations(raw, verbose=False)
                raw.load_data(verbose=False)
                raw.filter(l_freq=L_FREQ, h_freq=H_FREQ, verbose=False)
                raw.set_eeg_reference('average', projection=False, verbose=False)

                sfreq = raw.info['sfreq']
                event_map = {
                    event_id['TASK4T0']: 1,
                    event_id['TASK4T1']: 2,
                    event_id['TASK4T2']: 3
                }

                windows, window_samples, step_samples, total_samples = create_sliding_windows(
                    raw, window_size, step_size
                )
                picks = mne.pick_channels(raw.ch_names, CHANNELS)
                windows = windows[:, picks, :]

                y, groups = create_window_labels(
                    events, event_map, total_samples, window_samples, step_samples,
                    threshold=label_threshold, return_groups=True
                )
                groups = np.where(groups >= 0, groups + trial_offset, -1)
                trial_offset += len(events)

                # Covarianze su TUTTE le finestre: il baricentro va stimato sull'intera run,
                # senza guardare quali finestre siano attive ne' di che classe siano.
                covs = run_covariances(windows, sfreq, BANDS)

                mask_active = y != 1
                per_run[run] = {
                    "covs": covs,
                    "mask": mask_active,
                    "y": np.where(y[mask_active] == 2, 0, 1),
                    "groups": groups[mask_active],
                }

            except Exception as e:
                print(f"Errore {subject} run {run}: {e}")

        if not all(r in per_run for r in runs):
            print(f"  Dati insufficienti per la run di test {test_run}")
            continue

        # --- Riallineamento ---------------------------------------------------
        # Se il riferimento e' quello di training, si calcola una volta sola dalle covarianze
        # delle run di training messe insieme; altrimenti ogni run usa il proprio.
        if RECENTER == "train":
            references = [
                reference_mean(np.concatenate([per_run[r]["covs"][b] for r in train_runs]))
                for b in range(len(BANDS))
            ]

        features = {}
        for run in runs:
            aligned = []
            for b in range(len(BANDS)):
                covs_b = per_run[run]["covs"][b]
                ref = references[b] if RECENTER == "train" else reference_mean(covs_b)
                aligned.append(recenter(covs_b, ref))

            # Solo dopo il riallineamento si tengono le finestre attive
            mask = per_run[run]["mask"]
            features[run] = to_tangent([a[mask] for a in aligned])

        X_train = np.concatenate([features[r] for r in train_runs])
        y_train = np.concatenate([per_run[r]["y"] for r in train_runs])
        groups_train = np.concatenate([per_run[r]["groups"] for r in train_runs])

        X_test = features[test_run]
        y_test = per_run[test_run]["y"]

        # --- Classificazione ---------------------------------------------------
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearDiscriminantAnalysis(solver="lsqr")),
        ])

        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
        grid = GridSearchCV(pipe, param_grid, cv=cv,
                            scoring="balanced_accuracy", n_jobs=-1)
        grid.fit(X_train, y_train, groups=groups_train)

        probs = grid.predict_proba(X_test)
        max_probs = np.max(probs, axis=1)
        predictions = np.argmax(probs, axis=1)

        threshold = START_THRESHOLD
        accepted_mask = max_probs >= threshold
        while np.sum(accepted_mask) < MIN_ACCEPTED_RATIO * len(y_test) and threshold > MIN_THRESHOLD:
            threshold -= 0.05
            accepted_mask = max_probs >= threshold

        accepted = int(np.sum(accepted_mask))
        total = len(y_test)
        discarded = total - accepted

        accuracy = balanced_accuracy_score(y_test[accepted_mask], predictions[accepted_mask])
        all_accuracy[i - first_person, test_index] = accuracy
        all_discarded[i - first_person, test_index] = 100 * discarded / total
        cm_sum += confusion_matrix(y_test[accepted_mask], predictions[accepted_mask])

        print(f"  Run test: {test_run} | Threshold: {threshold:.2f} | "
              f"Riallineamento: {RECENTER}")
        print(f"  Campioni: {total} tot / {accepted} accettati / {discarded} scartati "
              f"({100*discarded/total:.1f}%)")
        print(f"  Balanced accuracy sugli accettati: {accuracy*100:.2f}% | "
              f"score interno: {grid.best_score_*100:.2f}% | shrinkage: {grid.best_params_}")
        print(f"  Tempo: {time.time() - test_start:.1f}s\n")

    patient_mean = np.nanmean(all_accuracy[i - first_person])
    patient_discarded = np.nanmean(all_discarded[i - first_person])
    print(f"  Paziente {subject}: {patient_mean*100:.2f}% con {patient_discarded:.1f}% di scarti "
          f"| Tempo: {time.time() - subject_start:.1f}s\n")

# --- Riepilogo ---------------------------------------------------------------

n_folds = int(np.count_nonzero(~np.isnan(all_accuracy)))
n_falliti = all_accuracy.size - n_folds
print("=" * 60)
print(f"Fold riusciti:          {n_folds}/{all_accuracy.size}"
      + (f" ({n_falliti} falliti, esclusi dalle medie)" if n_falliti else ""))
print(f"Accuratezza media:      {np.nanmean(all_accuracy)*100:.2f}%")
print(f"Deviazione standard:    {np.nanstd(all_accuracy)*100:.2f}%")
print(f"Campioni scartati:      {np.nanmean(all_discarded):.1f}% in media")
print(f"Matrice di confusione media:\n{(cm_sum / n_folds).astype(int)}")


Feature per finestra: 135 (3 bande x 9 canali)

Paziente 001
  Run test: 14 | Threshold: 0.90 | Riallineamento: per_run
  Campioni: 84 tot / 74 accettati / 10 scartati (11.9%)
  Balanced accuracy sugli accettati: 89.18% | score interno: 90.56% | shrinkage: {'clf__shrinkage': 0.5}
  Tempo: 17.5s

  Run test: 10 | Threshold: 0.90 | Riallineamento: per_run
  Campioni: 84 tot / 79 accettati / 5 scartati (6.0%)
  Balanced accuracy sugli accettati: 91.27% | score interno: 82.22% | shrinkage: {'clf__shrinkage': 'auto'}
  Tempo: 11.1s

  Run test: 6 | Threshold: 0.90 | Riallineamento: per_run
  Campioni: 84 tot / 78 accettati / 6 scartati (7.1%)
  Balanced accuracy sugli accettati: 88.46% | score interno: 85.28% | shrinkage: {'clf__shrinkage': 'auto'}
  Tempo: 7.4s

  Paziente 001: 89.64% con 8.3% di scarti | Tempo: 36.0s

Paziente 002
  Run test: 14 | Threshold: 0.90 | Riallineamento: per_run
  Campioni: 84 tot / 77 accettati / 7 scartati (8.3%)
  Balanced accuracy sugli accettati: 94.77% | s

C:\Users\papar\AppData\Roaming\Python\Python314\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  Run test: 10 | Threshold: 0.90 | Riallineamento: per_run
  Campioni: 84 tot / 77 accettati / 7 scartati (8.3%)
  Balanced accuracy sugli accettati: 77.64% | score interno: 78.61% | shrinkage: {'clf__shrinkage': 0.2}
  Tempo: 810.4s

  Run test: 6 | Threshold: 0.90 | Riallineamento: per_run
  Campioni: 84 tot / 70 accettati / 14 scartati (16.7%)
  Balanced accuracy sugli accettati: 71.43% | score interno: 80.28% | shrinkage: {'clf__shrinkage': 'auto'}
  Tempo: 12.6s

  Paziente 106: 76.72% con 18.3% di scarti | Tempo: 833.0s

Paziente 107
  Run test: 14 | Threshold: 0.90 | Riallineamento: per_run
  Campioni: 84 tot / 67 accettati / 17 scartati (20.2%)
  Balanced accuracy sugli accettati: 66.83% | score interno: 51.67% | shrinkage: {'clf__shrinkage': 0.5}
  Tempo: 10.6s

  Run test: 10 | Threshold: 0.90 | Riallineamento: per_run
  Campioni: 84 tot / 61 accettati / 23 scartati (27.4%)
  Balanced accuracy sugli accettati: 52.75% | score interno: 71.39% | shrinkage: {'clf__shrinkage': 0.5